### Filtragem de Questões do ENEM com base na API

Na API a ordem segue a do caderno Azul.

In [2]:
import pandas as pd
import json
from pprint import pprint
import os
import re
import numpy as np
import matplotlib.pyplot as plt

#### Modularização da Limpeza

In [3]:
def load_json(caminho):
    with open(caminho, 'r', encoding='utf-8') as f:
        return json.load(f)

def filtrar_questoes(data, disciplina="linguagens"):
    questions = data.get("questions", data)

    filtradas = [
        item for item in questions
        if (
            item.get("files", []) == [] and
            item.get("language") is None and
            item.get("discipline", "").lower() == disciplina.lower()
        )
    ]

    return filtradas

def limpar_texto(texto): #limpeza de texto
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = re.sub(r"\*\*", "", texto)
    texto = texto.replace("\xa0", " ")
    texto = texto.replace("/", " ")
    texto = texto.replace("\n", " ")
    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()

def extrair_alternativas(lista):
    resultado = {"A": None, "B": None, "C": None, "D": None, "E": None}

    if isinstance(lista, list):
        for alt in lista:
            letra = alt.get("letter")
            texto = limpar_texto(alt.get("text"))
            if letra in resultado:
                resultado[letra] = texto

    return pd.Series(resultado)

def transformar_dataframe(filtered):
    df = pd.json_normalize(filtered)

    # limpar textos
    df["context"] = df["context"].apply(limpar_texto)
    df["alternativesIntroduction"] = df["alternativesIntroduction"].apply(limpar_texto)

    # montar pergunta
    df["question"] = (
        df["context"].fillna("") + " " +
        df["alternativesIntroduction"].fillna("")
    ).apply(limpar_texto)

    # extrair alternativas
    alts = df["alternatives"].apply(extrair_alternativas)
    df = pd.concat([df, alts], axis=1)

    # dataset final
    df_final = df[[
        "index", "question",
        "A", "B", "C", "D", "E",
        "correctAlternative"
    ]].rename(columns={
        "index": "id",
        "correctAlternative": "correct"
    })

    df_final = df_final.reset_index(drop=True)

    return df_final

def montar_opcoes(row):
    return [
        f"A. {row['A']}",
        f"B. {row['B']}",
        f"C. {row['C']}",
        f"D. {row['D']}",
        f"E. {row['E']}",
    ]

def gerar_dataset_json(df_final):
    df_final["options"] = df_final.apply(montar_opcoes, axis=1)

    dataset = []

    for _, row in df_final.iterrows():
        dataset.append({
            "id": int(row["id"]),
            "question": row["question"],
            "options": row["options"]
        })

    return dataset

def pipeline_enem(caminho):
    data = load_json(caminho)
    filtradas = filtrar_questoes(data)
    df_final = transformar_dataframe(filtradas)
    dataset = gerar_dataset_json(df_final)

    return df_final, dataset

def exportar_json(dataset, caminho_saida):
    with open(caminho_saida, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

In [ ]:
def carregar_itens(caminho):
    return pd.read_csv(caminho, sep=';', encoding='latin1')

def filtrar_itens(itens, area='LC', cor='AZUL', co_prova=None):
    df = itens.copy()
    if area:
        df = df[df["SG_AREA"] == area]
    if cor:
        df = df[df["TX_COR"] == cor]
    if co_prova:
        df = df[df["CO_PROVA"] == co_prova]
    return df

def cruzar_itens_com_dataset(itens_df, df_final):
    return itens_df[itens_df["CO_POSICAO"].isin(df_final["id"])]

#### ENEM 2022 - LC

In [12]:
caminho_2022 = r"D:\ODG\outputs n8n\API Response\questões_enem_2022_lc.json"
df2022, dataset_json2022 = pipeline_enem(caminho_2022)
print(df2022.shape)
df2022

(31, 9)


,id,question,A,B,C,D,E,correct,options
0,6,"Urgência emocional Se tudo é para ontem, se a ...","Impessoalização ao longo do texto, como em: “s...","Construção de uma atmosfera de urgência, em pa...",Repetição de uma determinada estrutura sintáti...,"Ênfase no emprego da hipérbole, como em: “uma ...","Emprego de metáforas, como em: “a vida engata ...",E,"[A. Impessoalização ao longo do texto, como em..."
1,8,É ruivo? Tem olhos azuis? É homem ou mulher? U...,Contribuir para a formação cidadã dos jogadores.,Refutar modelos estereotipados de beleza e ele...,Estimular a competitividade entre potenciais c...,Exemplificar estratégias de arrecadação financ...,Desenvolver conhecimentos lúdicos específicos ...,A,[A. Contribuir para a formação cidadã dos joga...
2,10,"Ciente de que, no campo da criação, as inovaçõ...",Sucesso dos artistas.,Valorização dos suportes.,Proteção da produção estética.,Modo de distribuição de obras.,Compartilhamento das obras artísticas.,C,"[A. Sucesso dos artistas., B. Valorização dos ..."
3,11,"Ora, sempre que surge uma nova técnica, ela qu...",Dificuldade na apropriação da nova linguagem.,Valorização da utilização da nova tecnologia.,Recorrência das mudanças tecnológicas.,Suplantação imediata dos conhecimentos prévios.,Rapidez no aprendizado do manuseio das novas i...,A,[A. Dificuldade na apropriação da nova linguag...
4,12,Papos — Me disseram… — Disseram-me. — Hein? — ...,Falta de compreensão causada pelo choque entre...,Contexto de comunicação em que a conversa se dá.,Grau de polidez distinto entre os interlocutores.,Diferença de escolaridade entre os falantes.,Nível social dos participantes da situação.,B,[A. Falta de compreensão causada pelo choque e...
5,13,"São vários os fatores, internos e externos, qu...",Apontam o desenvolvimento econômico com soluçã...,Questionam a crença de que o acesso à informaç...,Afirmam que o uso comercial da rede é a causa ...,Refutam o vinculo entre níveis de escolaridade...,Condicionam a expansão da rede à elaboração de...,B,[A. Apontam o desenvolvimento econômico com so...
6,14,"TEXTO I A língua não é uma nomenclatura, que s...",Expediente próprio do sistema linguístico que ...,Ato inventivo de nomear novas realidades que s...,Mecanismo de apropriação de formas linguística...,Processo de incorporação de preconceitos que s...,Recurso de expressão marcado pela objetividade...,D,[A. Expediente próprio do sistema linguístico ...
7,16,"Ela era linda. Gostava de dançar, fazia teatro...","Narrar, por meio de imagem e poesia, cenas da ...","Descrever, por meio das memórias de Petra, a s...","Sintetizar, por meio das principais cenas do f...","Lançar, por meio da história de vida do autor,...","Avaliar, por meio de análise crítica, o filme ...",E,"[A. Narrar, por meio de imagem e poesia, cenas..."
8,17,PALAVRA As gramáticas classificam as palavras ...,"Tematiza o fazer poético, como em ""Os poetas c...","Utiliza o recurso expressivo da metáfora, como...","Valoriza a gramática da língua, como em ""subst...","Estabelece comparações, como em ""As palavras t...",Apresenta informações pertinentes acerca do co...,B,"[A. Tematiza o fazer poético, como em ""Os poet..."
9,18,Morte lenta ao luso infame que inventou a calç...,Justaposição de sequências verbais e nominais.,Mudança de eventos resultante do jogo temporal.,Uso de adjetivos qualificativos na descrição d...,Encadeamento semântico pelo uso de substantivo...,Inter-relação entre orações por elementos ling...,A,[A. Justaposição de sequências verbais e nomin...


In [5]:
dataset_json2022

[{'id': 6,
  'question': 'Urgência emocional Se tudo é para ontem, se a vida engata uma primeira e sai em disparada, se não há mais tempo para paradas estratégicas, caímos fatalmente no vício de querer que os amores sejam igualmente resolvidos num átimo de segundo. Temos pressa para ouvir “eu te amo”. Não vemos a hora de que fiquem estabelecidas as regras de convívio: somos namorados, ficantes, casados, amantes? Urgência emocional. Uma cilada. Associamos diversas palavras ao AMOR: paixão, romance, sexo, adrenalina, palpitação. Esquecemos, no entanto, da palavra que viabiliza esse sentimento: “paciência”. Amor sem paciência não vinga. Amor não pode ser mastigado e engolido com emergência, com fome desesperada. É uma refeição que pode durar uma vida. MEDEIROS, M. Disponível em: http: porumavidasimples.blogspot.com.br. Acesso em: 20 ago. 2017 (adaptado). Nesse texto de opinião, as marcas linguísticas revelam uma situação distensa e de pouca formalidade, o que se evidencia pelo(a)',
  'opt

In [28]:
itens2022 = carregar_itens(r'D:\ODG\ITENS_PROVA_2022.csv')

itens_filtrados2022 = filtrar_itens(
    itens2022,
    area='LC',
    cor='AZUL',
    co_prova=1065 #1179
)

itens_final2022 = cruzar_itens_com_dataset(itens_filtrados2022, df2022)
itens_final2022 = itens_final2022.sort_values("CO_POSICAO")
itens_final2022

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO
445,6,LC,60382,E,25,0,NaN,1.77346,0.68047,0.08704,AZUL,1065,NaN,0
443,8,LC,118120,A,23,0,NaN,1.53331,2.16698,0.23705,AZUL,1065,NaN,0
441,10,LC,49544,C,28,0,NaN,2.43974,1.15872,0.12513,AZUL,1065,NaN,0
440,11,LC,41810,A,29,0,NaN,0.92958,0.15096,0.01173,AZUL,1065,NaN,0
439,12,LC,42674,B,27,0,NaN,2.26956,0.51892,0.23370,AZUL,1065,NaN,0
438,13,LC,120622,B,30,0,NaN,2.55800,0.65099,0.12178,AZUL,1065,NaN,0
437,14,LC,42940,D,22,0,NaN,1.30734,-0.29879,0.00797,AZUL,1065,NaN,0
435,16,LC,141427,E,3,0,NaN,2.40522,2.28139,0.11370,AZUL,1065,NaN,0
434,17,LC,141464,B,4,0,NaN,0.94658,1.25172,0.21415,AZUL,1065,NaN,0
433,18,LC,67229,A,18,0,NaN,2.64295,1.76932,0.18786,AZUL,1065,NaN,0


In [26]:
#itens_azul = itens2022.query("SG_AREA == 'LC' and TX_COR == 'AZUL'")
#itens_azul
#itens_azul.query("CO_POSICAO == 6 and TX_GABARITO == 'E'")
#itens_azul.query("CO_POSICAO == 8 and TX_GABARITO == 'A'")
#itens_azul.query("CO_POSICAO == 45 and TX_GABARITO == 'E'")

#peguei duas questões e o gabarito delas pra ver qual é o co_prova que a API utiliza
# CO_PROVA é 1201

In [59]:
exportar_json(dataset_json2022, r"D:\ODG\outputs n8n\API Response\datasets\enem2022_lc.json")

In [30]:
#itens_final2022.to_csv("itens_enem_2022_azul.csv", index=False, encoding="utf-8")

In [56]:
#itens_final2022.to_excel("itens_enem_2022_azul.xlsx",index=False)

#### ENEM 2021 - LC

In [7]:
caminho_2021 = r"D:\ODG\outputs n8n\API Response\questoes_enem_2021_lc.json"
df2021, dataset_json2021 = pipeline_enem(caminho_2021)
print(df2021.shape)
dataset_json2021

(25, 9)


[{'id': 6,
  'question': 'Um asteroide de cerca de um mil metros diâmetro, viajando a 288 mil quilômetros por hora, passou a uma distância insignificante – em termos cósmicos – da Terra, pouco mais do dobre da distância que nos separa da Lua. Segundo os cálculos matemáticos, o asteroide cruzou a órbita da Terra e somente não colidiu porque ela não estava naquele ponto de interseção. Se ele tivesse sido capturado pelo campo gravitacional do nosso planeta e colidido, o impacto equivaleria a 40 bilhões de toneladas de TNT, ou o equivalente à explosão de 40 mil bombas de hidrogênio, conforme calcularam os computadores operados pelos astrônomos do programa de Exploração do Sistema Solar da Nasa; se caísse no continente, abriria uma cratera de cinco quilômetros, no mínimo, e destruiria tudo o que houvesse num raio de milhares de outros; se desabasse no oceano, provocaria maremotos que devastariam imensas regiões costeiras. Enfim, uma visão do Apocalipse. Disponível em: http: bdjur.stj.jus.br

In [33]:
df2021

,id,question,A,B,C,D,E,correct,options
0,6,Um asteroide de cerca de um mil metros diâmetr...,A descrição da velocidade do asteroide.,A recorrência de formulações hipotéticas.,A referência à opinião dos astrônomos.,"A utilização da locução adverbial ""no mínimo"".",A comparação com a distância da Lua à Terra.,B,"[A. A descrição da velocidade do asteroide., B..."
1,7,A draga A gente não sabia se aquela draga tinh...,Contrapõe características da escrita e da fala.,Ironiza a comunicação fora da norma-padrão.,Substitui regionalismos por registros formais.,Valoriza o uso de variedades populares.,Defende novas regras gramaticais.,D,[A. Contrapõe características da escrita e da ...
2,8,"O documentário _O menino que fez um museu_, di...",Originalidade da iniciativa de homenagem à vid...,Falta de concorrentes ao prêmio de uma das ass...,Proeza da premiação de uma história ambientada...,Escassez de investimentos para a produção cine...,Importância da parceria entre brasileiros e br...,C,[A. Originalidade da iniciativa de homenagem à...
3,10,"Intenso e original, Son of Saul retrata horror...","""[...] do estreante em longa-metragens László ...","""Ele é Saul (Géza Rohring), um dos encarregado...","""[...] a câmera está o tempo todo com o person...","""Saul percorre diferentes divisões de Auschwit...","""[...] premiar uma abordagem tão ousada e radi...",E,"[A. ""[...] do estreante em longa-metragens Lás..."
4,11,Sinhá Se a dona se banhou Eu não estava lá Por...,Remetem à violência física e simbólica contra ...,Valorizam as influências da cultura africana s...,Relativizam o sincretismo constitutivo das prá...,Narram os infortúnios da relação amorosa entre...,Problematizam as diferentes visões de mundo na...,A,[A. Remetem à violência física e simbólica con...
5,15,"TEXTO I Correu à sala dos retratos, abriu o pi...",Pouca complexidade musical das composições aju...,Prevalência de referências musicais africanas ...,Incipiente atribuição de prestígio social a mú...,Tensa relação entre o erudito e o popular na c...,Importância atribuída à música clássica a soci...,E,[A. Pouca complexidade musical das composições...
6,16,"Devagar, devagarinho Desacelerar é preciso. Ac...",Adverte sobre os riscos que o ritmo acelerado ...,Exemplifica o fato criticado no texto com uma ...,Contrapõe situações de aceleração e de serenid...,Questiona o clichê sobre a rapidez e a acelera...,Apresenta soluções para a cultura da correria ...,B,[A. Adverte sobre os riscos que o ritmo aceler...
7,17,"A história do futebol brasileiro contém, ao lo...",Responsabilização dos jogadores negros pela de...,Projeção mundial da nação por um esporte antes...,Depreciação de um esporte associado a marginal...,"Interdição da palavra ""racismo"" no contexto es...","Atitude contestadora de um ""jogador-problema"".",E,[A. Responsabilização dos jogadores negros pel...
8,19,Coincidindo com o Dia Internacional dos Direit...,Fomentaram as relações sociais entre as crianças.,Tornaram o lazer uma prática difundida entre a...,Incentivaram a criação de novos espaços para s...,Promoveram uma vivência corporal menor ativa.,Contribuíram para o aumento do tempo dedicado ...,D,[A. Fomentaram as relações sociais entre as cr...
9,20,O coreógrafo e bailarino Didier Mulleras se de...,Adotar uma perspectiva conceitual como contrap...,Criar novas formas de financiamento ao utiliza...,Privilegiar movimentos gerados por computação ...,"Produzir uma arte multimodal, com o intuito de...",Redefinir a extensão e o propósito do espetácu...,D,[A. Adotar uma perspectiva conceitual como con...


In [ ]:
itens2021 = carregar_itens(r'D:\ODG\ITENS_PROVA_2021.csv')

itens_filtrados2021 = filtrar_itens(
    itens2021,
    area='LC',
    cor='AZUL',
    co_prova=889 #1003
)

itens_final2021 = cruzar_itens_com_dataset(itens_filtrados2021, df2021)
itens_final2021 = itens_final2021.sort_values("CO_POSICAO")
itens_final2021

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO
2619,6,LC,15069,B,1.0,0,NaN,1.77244,0.23114,0.00953,AZUL,889,NaN,NaN
2620,7,LC,120165,D,25.0,0,NaN,2.95920,0.57101,0.16401,AZUL,889,NaN,NaN
2621,8,LC,118073,C,24.0,0,NaN,1.68791,0.79504,0.13105,AZUL,889,NaN,NaN
2623,10,LC,97519,E,18.0,0,NaN,2.39514,-0.22063,0.05035,AZUL,889,NaN,NaN
2624,11,LC,68920,A,20.0,0,NaN,0.96148,-0.66104,0.01713,AZUL,889,NaN,NaN
2628,15,LC,120646,D,14.0,0,NaN,1.57672,1.15232,0.16243,AZUL,889,NaN,NaN
2629,16,LC,86515,B,24.0,0,NaN,2.19688,0.92018,0.10506,AZUL,889,NaN,NaN
2630,17,LC,120170,E,9.0,0,NaN,1.98660,1.65243,0.11374,AZUL,889,NaN,NaN
2632,19,LC,118218,D,9.0,0,NaN,2.68787,0.62089,0.26778,AZUL,889,NaN,NaN
2633,20,LC,120623,D,30.0,0,NaN,1.58074,1.24997,0.12025,AZUL,889,NaN,NaN


In [ ]:
#itens_azul = itens2021.query("SG_AREA == 'LC' and TX_COR == 'AZUL'")
#itens_azul.query("CO_POSICAO == 6 and TX_GABARITO == 'B'")
#itens_azul.query("CO_POSICAO == 7 and TX_GABARITO == 'D'")
#itens_azul.query("CO_POSICAO == 8 and TX_GABARITO == 'C'")
#itens_azul

In [61]:
exportar_json(dataset_json2021, r"D:\ODG\outputs n8n\API Response\datasets\enem2021_lc.json")

In [ ]:
#itens_final2021.to_csv("itens_enem_2021_azul.csv", index=False, encoding="utf-8")

In [54]:
#itens_final2021.to_excel("itens_enem_2021_azul.xlsx",index=False)

#### ENEM 2020 - LC

In [9]:
caminho_2020 = r"D:\ODG\outputs n8n\API Response\questoes_enem_2020_lc.json"
df2020, dataset_json2020 = pipeline_enem(caminho_2020)
print(df2020.shape)
dataset_json2020

(29, 9)


[{'id': 6,
  'question': '_Vou-me embora p’ra Pasárgada_ foi o poema de mais longa gestação em toda a minha obra. Vi pela primeira vez esse nome Pasárgada quando tinha os meus dezesseis anos e foi num a autor grego. \\[…\\] Esse nome de Pasárgada, que significa “campo dos persas” ou “tesouro dos persas”, suscitou na minha imaginação uma paisagem fabulosa, um país de delícias, como o de _L’invitation au Voyage_, de Baudelaire. Mais de vinte anos depois, quando eu morava só na minha casa da Rua do Curvelo, num momento de fundo desânimo, da mais aguda sensação de tudo o que eu não tinha feito em minha vida por motivo da doença, saltou-me de súbito do subconsciente este grito estapafúrdio: “Vou-me embora p’ra Pasárgada!” Senti na redondilha a primeira célula de um poema, e tentei realizá-lo, mas fracassei. Alguns anos depois, em idênticas circunstâncias de desalento e tédio, me ocorreu o mesmo desabafo de evasão da “vida besta”. Desta vez o poema saiu sem esforço como se já estivesse pront

In [42]:
df2020

,id,question,A,B,C,D,E,correct,options
0,6,_Vou-me embora p’ra Pasárgada_ foi o poema de ...,"Emotiva, porque o poeta expõe os sentimentos d...","Referencial, porque o texto informa sobre a or...","Metalinguística, porque o poeta tece comentári...","Poética, porque o texto aborda os elementos es...","Apelativa, porque o poeta tenta convencer os l...",C,"[A. Emotiva, porque o poeta expõe os sentiment..."
1,8,_Slam_ do Corpo é um encontro pensado para sur...,Imprimir ritmo e visibilidade à expressão poét...,Redefinir o espaço de circulação da poesia urb...,Estimular produções autorais de usuários de Li...,Traduzir expressões verbais para a língua de s...,Proporcionar performances estéticas de pessoas...,A,[A. Imprimir ritmo e visibilidade à expressão ...
2,9,É possível afirmar que muitas expressões idiom...,Registros do inventário do português brasileiro.,Justificativas da variedade linguística do país.,Influências da fala do nordestino no uso da lí...,Explorações do falar de um grupo social especí...,Representações da mudança linguística do portu...,A,[A. Registros do inventário do português brasi...
3,10,"O ouro do século 21 Cério, gadolínio, lutécio,...",Imprimir um tom irônico à reportagem.,Incorporar citações de especialistas à reporta...,"Atribuir maior valor aos metais, objeto da rep...",Esclarecer termos científicos empregados na re...,Marcar a apropriação de termos de outra ciênci...,C,"[A. Imprimir um tom irônico à reportagem., B. ..."
4,11,Na sua imaginação perturbada sentia a natureza...,Relação entre a natureza opressiva e o desejo ...,Confluência entre o estado emocional da person...,Prevalência do mundo natural em relação à frag...,Depreciação do sentido da vida diante da consc...,Instabilidade psicológica da personagem face à...,B,[A. Relação entre a natureza opressiva e o des...
5,12,"TEXTO I É pau, é pedra, é o fim do caminho É u...",Diálogo que a letra da canção estabelece com d...,Singularidade com que o compositor converte re...,Caráter inovador com que o compositor concebe ...,Relativização que a letra da canção promove na...,O resgate que a letra da canção promove de obr...,A,[A. Diálogo que a letra da canção estabelece c...
6,13,"Deu vontade de jogar, mas não sabe como reunir...",Organização de eventos de competições esportivas.,Agendamento de viagens para eventos de esporte...,Mapeamento dos interesses dos praticantes acer...,Identificação da escassez de espaços para a vi...,Formação de grupos em comunidades virtuais par...,E,[A. Organização de eventos de competições espo...
7,16,"Fomos falar com o tal encarregado, depois com ...",Sequência da atribuição de responsabilidades e...,Solicitação em nome dos prejuízos e compromiss...,Intimidação pela discreta presença de um agent...,Promessa de imediato atendimento da carência s...,Apelo pela identificação com a empresa extensi...,E,[A. Sequência da atribuição de responsabilidad...
8,17,Uma das mais contundentes críticas ao discurso...,Constrói a ideia de que a mudança individual d...,Considera a homogeneidade da escolha de hábito...,Reforça a necessidade de solucionar os problem...,Problematiza a organização social e seu impact...,Reproduz a noção de que a melhoria da aptidão ...,D,[A. Constrói a ideia de que a mudança individu...
9,18,Retrato de homem A paisagem estrita ao apuro d...,Desvela sentimentos de vazio e angústia sob a ...,Expressa desilusão ante a possibilidade de sup...,Contrapõe a fragilidade emocional ao uso desme...,Associa a incomunicabilidade emocional às dete...,Privilegia imagens relacionadas à exposição do...,A,[A. Desvela sentimentos de vazio e angústia so...


In [47]:
itens2020 = carregar_itens(r'D:\ODG\ITENS_PROVA_2020.csv')

itens_filtrados2020 = filtrar_itens(
    itens2020,
    area='LC',
    cor='AZUL',
    co_prova=577 #1003
)

itens_final2020 = cruzar_itens_com_dataset(itens_filtrados2020, df2020)
itens_final2020 = itens_final2020.sort_values("CO_POSICAO")
itens_final2020

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO,TP_VERSAO_DIGITAL
1787,6,LC,63448,C,19,0,NaN,3.18825,1.25880,0.17432,AZUL,577,NaN,0,NaN
484,8,LC,21377,A,14,0,NaN,1.72932,1.85051,0.14869,AZUL,577,NaN,0,NaN
4358,9,LC,112138,A,20,0,NaN,1.52123,1.91811,0.18523,AZUL,577,NaN,0,NaN
4200,10,LC,112059,C,23,0,NaN,1.10742,1.64000,0.25509,AZUL,577,NaN,0,NaN
3326,11,LC,96433,B,16,0,NaN,1.93001,1.02406,0.14189,AZUL,577,NaN,0,NaN
4304,12,LC,112112,A,20,0,NaN,1.74167,1.51202,0.24386,AZUL,577,NaN,0,NaN
4275,13,LC,112089,E,28,0,NaN,1.60955,0.26979,0.20531,AZUL,577,NaN,0,NaN
3372,16,LC,96861,E,17,0,NaN,2.41993,1.10401,0.14865,AZUL,577,NaN,0,NaN
1634,17,LC,59955,D,10,0,NaN,1.68626,0.30607,0.14643,AZUL,577,NaN,0,NaN
361,18,LC,17055,A,16,0,NaN,1.35608,1.44454,0.23512,AZUL,577,NaN,0,NaN


In [ ]:
#itens_azul = itens2020.query("SG_AREA == 'LC' and TX_COR == 'AZUL'")
#itens_azul.query("CO_POSICAO == 6 and TX_GABARITO == 'C'") 577
#itens_azul.query("CO_POSICAO == 8 and TX_GABARITO == 'A'") 577
#itens_azul.query("CO_POSICAO == 18 and TX_GABARITO == 'A'") 577
#itens_azul

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO,TP_VERSAO_DIGITAL
361,18,LC,17055,A,16,0,NaN,1.35608,1.44454,0.23512,AZUL,577,NaN,0,NaN


In [60]:
exportar_json(dataset_json2020, r"D:\ODG\outputs n8n\API Response\datasets\enem2020_lc.json")

In [49]:
#itens_final2020.to_csv("itens_enem_2020_azul.csv", index=False, encoding="utf-8")

In [52]:
#itens_final2020.to_excel("itens_enem_2020_azul.xlsx",index=False)

#### Formatação Antiga

### Questões do ENEM 2023

In [64]:
caminho = r"D:\ODG\API Response.json"
with open(caminho, 'r', encoding='utf-8') as f:
    data = json.load(f)

print("Chaves principais:", data.keys())

if "questions" in data:
    df = pd.json_normalize(data["questions"])
else:
    df = pd.json_normalize(data)

print("\nColunas do DataFrame:")
print(df.columns.tolist())

df

Chaves principais: dict_keys(['metadata', 'questions'])

Colunas do DataFrame:
['title', 'index', 'discipline', 'language', 'year', 'context', 'files', 'correctAlternative', 'alternativesIntroduction', 'alternatives']


,title,index,discipline,language,year,context,files,correctAlternative,alternativesIntroduction,alternatives
0,Questão 1 - ENEM 2023,1,linguagens,espanhol,2023,**TEXTO I**\n\n¿QUÉ ME PASA?: \n¿PorQUE ME CU...,[],A,O filme Como estrellas en la tierra aborda o t...,"[{'letter': 'A', 'text': 'Olhar diferenciado p..."
1,Questão 2 - ENEM 2023,2,linguagens,espanhol,2023,None,[],A,"Me niego rotundamente\nA negar mi voz,\nMi san...","[{'letter': 'A', 'text': 'Advérbios como ""rotu..."
2,Questão 3 - ENEM 2023,3,linguagens,espanhol,2023,**“Caramelos” en sus suelos**\n\nLas tierras d...,[],E,"Nesse poema, o eu poético enaltece a","[{'letter': 'A', 'text': 'Característica amist..."
3,Questão 4 - ENEM 2023,4,linguagens,espanhol,2023,Manipular es sembrar en la conciencia y en la ...,[],B,"Nesse texto, a expressão “cortina de humo” rev...","[{'letter': 'A', 'text': 'Amadurece tardiament..."
4,Questão 5 - ENEM 2023,5,linguagens,espanhol,2023,**Que quede claro**\n\nCómo es posible que se ...,[],A,"A letra da canção Que quede claro, da banda cu...","[{'letter': 'A', 'text': 'Indignação diante do..."
5,Questão 6 - ENEM 2023,6,linguagens,None,2023,None,[],D,A sessão do Comitê Olímpico Internacional (COI...,"[{'letter': 'A', 'text': 'Unificação do lema a..."
6,Questão 7 - ENEM 2023,7,linguagens,None,2023,**Mais iluminada que outras**\n\nTenho dois se...,[],A,"Nesse texto, os recursos expressivos usados pe...","[{'letter': 'A', 'text': 'Revelam as marcas da..."
7,Questão 8 - ENEM 2023,8,linguagens,None,2023,**De quem é esta língua?**\n\nUma pequena edit...,[],C,O texto de Agualusa tematiza o preconceito em ...,"[{'letter': 'A', 'text': 'À dificuldade de con..."
8,Questão 9 - ENEM 2023,9,linguagens,None,2023,"Na Idade Média, as notícias se propagavam com ...",[],E,A propagação sistemática de informações é um f...,"[{'letter': 'A', 'text': 'Velocidade de circul..."
9,Questão 10 - ENEM 2023,10,linguagens,None,2023,Se a interferência de contas falsas em discuss...,[],E,"De acordo com o texto, a análise de caracterís...","[{'letter': 'A', 'text': 'Controle da atuação ..."


In [65]:
questions = data["questions"]

print("Total de itens:", len(questions))

filtered = [
    item for item in questions
    if item.get("files", []) == [] and item.get("language") is None
]

print("Total após filtro:", len(filtered))

df_filtered = pd.json_normalize(filtered)

df_filtered = df_filtered[df_filtered["index"] != 44] #questão com imagem no contexto
df_filtered

Total de itens: 44
Total após filtro: 37


,title,index,discipline,language,year,context,files,correctAlternative,alternativesIntroduction,alternatives
0,Questão 6 - ENEM 2023,6,linguagens,None,2023,None,[],D,A sessão do Comitê Olímpico Internacional (COI...,"[{'letter': 'A', 'text': 'Unificação do lema a..."
1,Questão 7 - ENEM 2023,7,linguagens,None,2023,**Mais iluminada que outras**\n\nTenho dois se...,[],A,"Nesse texto, os recursos expressivos usados pe...","[{'letter': 'A', 'text': 'Revelam as marcas da..."
2,Questão 8 - ENEM 2023,8,linguagens,None,2023,**De quem é esta língua?**\n\nUma pequena edit...,[],C,O texto de Agualusa tematiza o preconceito em ...,"[{'letter': 'A', 'text': 'À dificuldade de con..."
3,Questão 9 - ENEM 2023,9,linguagens,None,2023,"Na Idade Média, as notícias se propagavam com ...",[],E,A propagação sistemática de informações é um f...,"[{'letter': 'A', 'text': 'Velocidade de circul..."
4,Questão 10 - ENEM 2023,10,linguagens,None,2023,Se a interferência de contas falsas em discuss...,[],E,"De acordo com o texto, a análise de caracterís...","[{'letter': 'A', 'text': 'Controle da atuação ..."
5,Questão 11 - ENEM 2023,11,linguagens,None,2023,"Maio foi colorido de amarelo, e o foi porque m...",[],D,Considerando os procedimentos argumentativos u...,"[{'letter': 'A', 'text': 'Enumerar as causas d..."
6,Questão 12 - ENEM 2023,12,linguagens,None,2023,Ainda daquela vez pude constatar a bizarrice d...,[],C,"Pela voz de uma empregada da casa, a descrição...","[{'letter': 'A', 'text': 'Opção por termos e e..."
7,Questão 13 - ENEM 2023,13,linguagens,None,2023,Girassol da madrugada \nTeu dedo curioso me s...,[],C,"Perante o outro, o eu lírico revela, na força ...","[{'letter': 'A', 'text': 'Vergonha das marcas ..."
8,Questão 14 - ENEM 2023,14,linguagens,None,2023,"**Dão Lalalão** \nDo povoado do Ão, ou dos sí...",[],D,"Nesse trecho do conto, o gosto dos moradores d...","[{'letter': 'A', 'text': 'Qualidade do som do ..."
9,Questão 15 - ENEM 2023,15,linguagens,None,2023,"As cinzas do Museu Nacional, no Rio de Janeiro...",[],B,A perda dos registros linguísticos no incêndio...,"[{'letter': 'A', 'text': 'Exige a retomada das..."


In [66]:
def limpar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto)
    # remover markdown ** **
    texto = re.sub(r"\*\*", "", texto)
    # remover non-breaking space
    texto = texto.replace("\xa0", " ")
    # remover barras
    texto = texto.replace("/", " ")
    # remover quebras de linha
    texto = texto.replace("\n", " ")
    # normalizar espaços
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


# --- dataframe base ---
df_new = df_filtered.copy()

# limpar textos principais
df_new["context"] = df_new["context"].apply(limpar_texto)
df_new["alternativesIntroduction"] = df_new["alternativesIntroduction"].apply(limpar_texto)

# --- juntar contexto + pergunta ---
df_new["question"] = (
    df_new["context"].fillna("") + " " + df_new["alternativesIntroduction"].fillna("")
)

df_new["question"] = df_new["question"].apply(limpar_texto)


# --- extrair alternativas ---
def extrair_alternativas(lista):

    resultado = {"A": None, "B": None, "C": None, "D": None, "E": None}

    if isinstance(lista, list):
        for alt in lista:
            letra = alt.get("letter")
            texto = limpar_texto(alt.get("text"))
            if letra in resultado:
                resultado[letra] = texto

    return pd.Series(resultado)


alts = df_new["alternatives"].apply(extrair_alternativas)

df_final = pd.concat([df_new, alts], axis=1)


# --- montar dataset final ---
df_final = df_final[[
    "index",
    "question",
    "A",
    "B",
    "C",
    "D",
    "E",
    "correctAlternative"
]]

df_final = df_final.rename(columns={
    "index": "id",
    "correctAlternative": "correct"
})


# reset index
df_final = df_final.reset_index(drop=True)

df_final

,id,question,A,B,C,D,E,correct
0,6,A sessão do Comitê Olímpico Internacional (COI...,Unificação do lema anterior ao atual.,Aproximação entre o lema olímpico e o COI.,Junção do lema olímpico com os princípios espo...,Associação entre o lema olímpico e a cooperati...,Vinculação entre o lema olímpico e os eventos ...,D
1,7,"Mais iluminada que outras Tenho dois seios, es...",Revelam as marcas da violência de raça e de gê...,Questionam o pioneirismo do estado do Ceará no...,Reproduzem padrões estéticos em busca da valor...,Sugerem uma atmosfera onírica alinhada ao dese...,"Mimetizam, na paisagem, os corpos transformado...",A
2,8,De quem é esta língua? Uma pequena editora bra...,À dificuldade de consolidação da literatura br...,Aos diferentes graus de instrução formal entre...,À existência de uma língua ideal que alguns fa...,Ao intercâmbio cultural que ocorre entre os po...,À distância territorial entre os falantes do p...,C
3,9,"Na Idade Média, as notícias se propagavam com ...",Velocidade de circulação das notícias.,Nível de letramento da população marginalizada.,Poder de censura por parte dos serviços públicos.,Legitimidade da voz dos representantes da nobr...,Diversidade dos meios disponíveis em uma época...,E
4,10,Se a interferência de contas falsas em discuss...,Controle da atuação dos profissionais de TI.,"Desenvolvimento de tecnologias como os ""trolls"".",Flexibilização dos turnos de trabalho dos cont...,Necessidade de regulamentação do funcionamento...,Identificação de padrões de disseminação de in...,E
5,11,"Maio foi colorido de amarelo, e o foi porque m...",Enumerar as causas determinantes da violência ...,Contextualizar a campanha de advertência no ce...,Divulgar dados numéricos alarmantes sobre acid...,Sensibilizar o público para a importância de u...,Restringir os problemas da violência no trânsi...,D
6,12,Ainda daquela vez pude constatar a bizarrice d...,Opção por termos e expressões de sentido ambíguo.,Crítica social inspirada pelo convívio com os ...,Descrição impressionista do fetiche do persona...,Presença de um foco narrativo de caráter impre...,Ambiência de mistério das relações entre famil...,C
7,13,Girassol da madrugada Teu dedo curioso me segu...,Vergonha das marcas provocadas pela passagem d...,Indecisão em face das possibilidades afetivas ...,Serenidade sedimentada pela entrega pacifica a...,Frustração causada pela vontade de retorno ao ...,Disponibilidade para a exploração do prazer ef...,C
8,14,"Dão Lalalão Do povoado do Ão, ou dos sítios pe...",Qualidade do som do rádio.,Estabilidade do enredo contado.,Ineditismo do capítulo da novela.,Jeito singular de falar aos ouvintes.,Dificuldade de compreensão da história.,D
9,15,"As cinzas do Museu Nacional, no Rio de Janeiro...",Exige a retomada das pesquisas por especialist...,Representa danos irreparáveis à memória e à id...,Impossibilita o surgimento de novas pesquisas ...,Resulta na extinção da cultura de povos origin...,Inviabiliza o estudo da língua do povo Tikuna.,B


In [77]:
def montar_opcoes(row):
    return [
        f"A. {row['A']}",
        f"B. {row['B']}",
        f"C. {row['C']}",
        f"D. {row['D']}",
        f"E. {row['E']}",
    ]

df_final["options"] = df_final.apply(montar_opcoes, axis=1)

dataset_json2023 = []

for _, row in df_final.iterrows():
    dataset_json2023.append({
        "id": int(row["id"]),
        "question": row["question"],
        "options": row["options"]
        #"correct": row["correct"]
    })

In [78]:
dataset_json2023

[{'id': 6,
  'question': 'A sessão do Comitê Olímpico Internacional (COI) aprovou uma mudança histórica e inédita no lema olímpico, criado em 1894 pelo Barão Pierre de Coubertin para expressar os valores e a excelência do esporte. Mais de 120 anos depois, o lema tem sua primeira alteração para ressaltar a solidariedade e incluir a palavra “juntos” mais rápido, mais alto, mais forte – juntos. A mudança foi aprovada por unanimidade pelos membros do COI e celebrada pelo presidente da entidade. Disponível em: https: ge.globo.com Acesso em: 10 nov. 2021 (adaptado) De acordo com o texto, a alteração do lema olímpico teve como objetivo a',
  'options': ['A. Unificação do lema anterior ao atual.',
   'B. Aproximação entre o lema olímpico e o COI.',
   'C. Junção do lema olímpico com os princípios esportivos.',
   'D. Associação entre o lema olímpico e a cooperatividade.',
   'E. Vinculação entre o lema olímpico e os eventos atléticos.']},
 {'id': 7,
  'question': 'Mais iluminada que outras Ten

#### Download do Dump

In [69]:
#with open("enem2023_lc.json", "w", encoding="utf-8") as f:
#    json.dump(dataset_json, f, ensure_ascii=False, indent=2)

### Itens e Cadernos do ENEM 2023

In [70]:
itens = pd.read_csv(r'D:\ODG\ITENS_PROVA_2023.csv', sep=';', encoding='latin1')
itens = itens.sort_values('CO_POSICAO')
itens.shape
itens

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO
1237,1,LC,140652,E,5,0,NaN,1.55494,0.70223,0.21103,LARANJA,1288,1.0,1
2842,1,LC,140652,E,5,0,NaN,1.55494,0.70223,0.21103,LEITOR TELA,1290,1.0,0
3992,1,LC,140765,A,6,0,NaN,3.17668,0.44857,0.18727,AZUL,1201,1.0,0
2903,1,LC,140595,B,5,0,NaN,2.11783,1.27877,0.12469,VERDE,1314,0.0,0
699,1,LC,140649,B,5,0,NaN,2.23798,0.86091,0.18548,VERDE,1314,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3649,180,MT,96099,B,1,0,NaN,2.10364,1.58069,0.19261,LARANJA,1298,NaN,1
4445,180,MT,96099,B,1,0,NaN,2.10364,1.58069,0.19261,AMARELA,1296,NaN,0
349,180,MT,125933,B,10,0,NaN,2.60187,2.11188,0.26394,VERDE,1219,NaN,0
4870,180,MT,85504,B,21,0,NaN,2.21375,1.82985,0.15414,AMARELA,1212,NaN,0


In [71]:
itens.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5550 entries, 1237 to 5086
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CO_POSICAO        5550 non-null   int64  
 1   SG_AREA           5550 non-null   object 
 2   CO_ITEM           5550 non-null   int64  
 3   TX_GABARITO       5550 non-null   object 
 4   CO_HABILIDADE     5550 non-null   int64  
 5   IN_ITEM_ABAN      5550 non-null   int64  
 6   TX_MOTIVO_ABAN    24 non-null     object 
 7   NU_PARAM_A        5526 non-null   float64
 8   NU_PARAM_B        5526 non-null   float64
 9   NU_PARAM_C        5526 non-null   float64
 10  TX_COR            5550 non-null   object 
 11  CO_PROVA          5550 non-null   int64  
 12  TP_LINGUA         300 non-null    float64
 13  IN_ITEM_ADAPTADO  5550 non-null   int64  
dtypes: float64(4), int64(6), object(4)
memory usage: 650.4+ KB


In [72]:
itens_azul = itens.query("SG_AREA == 'LC' and TX_COR == 'AZUL'")
itens_azul

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO
3992,1,LC,140765,A,6,0,NaN,3.17668,0.44857,0.18727,AZUL,1201,1.0,0
2994,1,LC,140718,E,8,0,NaN,2.73373,0.00161,0.19391,AZUL,1281,1.0,0
1138,1,LC,140730,D,7,0,NaN,2.77283,0.35669,0.17995,AZUL,1241,0.0,0
3844,1,LC,140743,B,6,0,NaN,2.79003,0.21669,0.49995,AZUL,1201,0.0,0
5063,1,LC,140730,D,7,0,NaN,2.77283,0.35669,0.17995,AZUL,1281,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3692,44,LC,140835,B,20,0,NaN,3.62150,-0.18111,0.18705,AZUL,1281,NaN,0
1835,44,LC,140835,B,20,0,NaN,3.62150,-0.18111,0.18705,AZUL,1241,NaN,0
2542,45,LC,140806,D,16,0,NaN,2.24767,0.37674,0.18584,AZUL,1241,NaN,0
1062,45,LC,140920,E,3,0,NaN,1.41999,0.67926,0.21271,AZUL,1201,NaN,0


In [73]:
itens_azul.query("CO_POSICAO == 6 and TX_GABARITO == 'D'")
#itens_azul.query("CO_POSICAO == 7 and TX_GABARITO == 'A'")

#peguei duas questões e o gabarito delas pra ver qual é o co_prova que a API utiliza
# CO_PROVA é 1201

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO
480,6,LC,141233,D,9,0,NaN,1.74151,0.50452,0.15417,AZUL,1201,NaN,0


In [74]:
ids = df_final["id"].tolist() #questões de LC

itens_azul = itens.query("CO_PROVA == 1201")
itens_azul = itens_azul[itens_azul["CO_POSICAO"].isin(df_final["id"])]
itens_azul

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO
480,6,LC,141233,D,9,0,NaN,1.74151,0.50452,0.15417,AZUL,1201,NaN,0
3991,7,LC,140884,A,17,0,NaN,1.35926,1.14954,0.23381,AZUL,1201,NaN,0
5341,8,LC,141443,C,27,0,NaN,2.20477,0.52119,0.13930,AZUL,1201,NaN,0
3312,9,LC,88031,E,30,0,NaN,1.73107,1.60103,0.19199,AZUL,1201,NaN,0
1206,10,LC,111887,E,29,0,NaN,3.04416,0.33951,0.18330,AZUL,1201,NaN,0
479,11,LC,118301,D,23,0,NaN,0.95292,-0.47764,0.00564,AZUL,1201,NaN,0
3990,12,LC,141776,C,15,0,NaN,1.22341,0.82185,0.06897,AZUL,1201,NaN,0
3311,13,LC,140949,C,16,0,NaN,1.43175,0.80537,0.12273,AZUL,1201,NaN,0
1205,14,LC,140899,D,29,0,NaN,1.22127,1.21426,0.22043,AZUL,1201,NaN,0
478,15,LC,118195,B,30,0,NaN,1.67063,-0.06011,0.08734,AZUL,1201,NaN,0


In [75]:
#itens_azul.to_csv("itens_enem_2023_azul.csv", index=False, encoding="utf-8")

In [76]:
itens_ordenados = itens_azul.sort_values(by="NU_PARAM_B")
itens_ordenados

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO
479,11,LC,118301,D,23,0,NaN,0.95292,-0.47764,0.00564,AZUL,1201,NaN,0
775,35,LC,141243,A,11,0,NaN,2.00614,-0.27647,0.09754,AZUL,1201,NaN,0
4532,28,LC,141448,C,19,0,NaN,0.99432,-0.13224,0.02226,AZUL,1201,NaN,0
3849,41,LC,141204,B,10,0,NaN,1.81453,-0.08926,0.19896,AZUL,1201,NaN,0
478,15,LC,118195,B,30,0,NaN,1.67063,-0.06011,0.08734,AZUL,1201,NaN,0
345,42,LC,141802,E,10,0,NaN,2.59779,-0.03089,0.21017,AZUL,1201,NaN,0
4952,19,LC,141474,D,23,0,NaN,1.63551,0.00891,0.20480,AZUL,1201,NaN,0
2471,38,LC,141214,A,13,0,NaN,2.36405,0.00914,0.19662,AZUL,1201,NaN,0
4533,27,LC,141814,A,18,0,NaN,1.22374,0.19619,0.20612,AZUL,1201,NaN,0
85,22,LC,140847,C,20,0,NaN,3.15133,0.25141,0.17380,AZUL,1201,NaN,0


### Filtrar questões

In [88]:
#def filtrar_questoes_por_id(dataset, ids):
#    return [item for item in dataset if item["id"] in ids]
def filtrar_com_ano(dataset, ids, ano):
    return [
        {**item, "ano": ano}
        for item in dataset
        if item["id"] in ids
    ]

In [89]:
q2023 = filtrar_com_ano(dataset_json2023, [9, 38], 2023)
q2022 = filtrar_com_ano(dataset_json2022, [26, 34, 31, 8], 2022)
q2021 = filtrar_com_ano(dataset_json2021, [11, 22, 17], 2021)
q2020 = filtrar_com_ano(dataset_json2020, [43, 36, 6], 2020)

dataset_final = q2023 + q2022 + q2021 + q2020

In [90]:
dataset_final

[{'id': 9,
  'question': 'Na Idade Média, as notícias se propagavam com surpreendente eficácia. Segundo uma emérita professora de Sorbonne, um cavalo era capaz de percorrer 30 quilômetros por dia, mas o tempo podia se acelerar dependendo do interesse da notícia. As ordens mendicantes tinham um papel importante na disseminação de informações, assim como os jograis, os peregrinos e os vagabundos, porque todos eles percorriam grandes distâncias. As cidades também tinham correios organizados e selos para lacrar mensagens e tentar certificar a veracidade das correspondências. Graças a tudo isso, a circulação de boatos era intensa e politicamente relevante. Um exemplo clássico de _fake news_ da era medieval é a história do rei que desaparece na batalha e reaparece muito depois, idoso e transformado. Disponível em: www.elpais.com.br. Acesso em: 18 jun. 2018 (adaptado) A propagação sistemática de informações é um fenômeno recorrente na história e no desenvolvimento das sociedades. No texto, a 

In [ ]:
#q2023 = filtrar_questoes_por_id(dataset_json2023, [9, 38])
#q2022 = filtrar_questoes_por_id(dataset_json2022, [26, 34, 31, 8])
#q2021 = filtrar_questoes_por_id(dataset_json2021, [11, 22, 17])
#q2020 = filtrar_questoes_por_id(dataset_json2020, [43, 36, 6])
#dataset_final = q2023 + q2022 + q2021 + q2020

In [82]:
dataset_final

[{'id': 9,
  'question': 'Na Idade Média, as notícias se propagavam com surpreendente eficácia. Segundo uma emérita professora de Sorbonne, um cavalo era capaz de percorrer 30 quilômetros por dia, mas o tempo podia se acelerar dependendo do interesse da notícia. As ordens mendicantes tinham um papel importante na disseminação de informações, assim como os jograis, os peregrinos e os vagabundos, porque todos eles percorriam grandes distâncias. As cidades também tinham correios organizados e selos para lacrar mensagens e tentar certificar a veracidade das correspondências. Graças a tudo isso, a circulação de boatos era intensa e politicamente relevante. Um exemplo clássico de _fake news_ da era medieval é a história do rei que desaparece na batalha e reaparece muito depois, idoso e transformado. Disponível em: www.elpais.com.br. Acesso em: 18 jun. 2018 (adaptado) A propagação sistemática de informações é um fenômeno recorrente na história e no desenvolvimento das sociedades. No texto, a 

In [91]:
print(len(dataset_final))  # deve dar 12

12


In [92]:
exportar_json(dataset_final, r"D:\ODG\outputs n8n\API Response\datasets\questoes_teste.json")